In [1]:
import numpy as np
import pandas as pd

In [3]:
df = pd.read_csv('covid_toy.csv')
df.head()

,age,gender,fever,cough,city,has_covid
0,60,Male,103.0,Mild,Kolkata,No
1,27,Male,100.0,Mild,Delhi,Yes
2,42,Male,101.0,Mild,Delhi,No
3,31,Female,98.0,Mild,Kolkata,No
4,65,Female,101.0,Mild,Mumbai,No


In [6]:
# if we observe above each column is different and for each column we have to apply different transformations
# for gendsum(er, city -> one hot encoding, for cough ordinal encoding, for fever imputaion
df.isnull().sum()

age           0
gender        0
fever        10
cough         0
city          0
has_covid     0
dtype: int64

In [9]:
from sklearn.model_selection import train_test_split
X_train, X_test, Y_train, Y_test = train_test_split(df.drop(columns = ['has_covid']), df['has_covid'], test_size = 0.2, random_state  =42)

In [10]:
# doing normal transformations 
# imputer for fever
from sklearn.impute import SimpleImputer

In [11]:
si = SimpleImputer()
X_train_fever = si.fit_transform(X_train[['fever']])

In [13]:
X_train_fever.shape

(80, 1)

In [17]:
X_test_fever = si.fit_transform(X_test[['fever']])

In [18]:
# ordinal encoding -> cough
from sklearn.preprocessing import OrdinalEncoder

In [20]:
oe = OrdinalEncoder(categories = [['Mild', 'Strong']])
X_train_cough = oe.fit_transform(X_train[['cough']])
X_test_cough = oe.fit_transform(X_test[['cough']])

In [26]:
# on hot encofding -> gener, city
from sklearn.preprocessing import OneHotEncoder
ohe = OneHotEncoder(drop = 'first', sparse_output = False)
X_train_gender_city = ohe.fit_transform(X_train[['gender', 'city']])
X_test_gender_city = ohe.fit_transform(X_test[['gender', 'city']])

In [23]:
# now getting only age
X_train_age = X_train.drop(columns = ['gender', 'fever', 'city', 'cough']).values
X_test_age = X_test.drop(columns = ['gender', 'fever', 'city', 'cough']).values

In [27]:
# now comibing all columns after feature trasnform happened
X_train_transformed = np.concatenate((X_train_age, X_train_fever, X_train_cough, X_train_gender_city), axis = 1)
X_test_transformed = np.concatenate((X_test_age, X_test_fever, X_test_cough, X_test_gender_city), axis = 1)

In [29]:
X_train_transformed.shape

(80, 7)

In [30]:
# if we use column transformer

In [32]:
from sklearn.compose import ColumnTransformer

In [38]:
transformer = ColumnTransformer(transformers = [
                ('tnf1', SimpleImputer(), ['fever']),
                ('tnf2', OrdinalEncoder(categories = [['Mild', 'Strong']]), ['cough']),
                ('tnf3', OneHotEncoder(sparse_output = False, drop = 'first',),['gender', 'city'])
], remainder = 'passthrough')

In [40]:
transformer.fit_transform(X_train).shape

(80, 7)